In [0]:
from pyspark.sql import functions as F
from src.transforms import validate_coordinates, add_silver_features

bronze_df = spark.table("bronze_ais_canonical")

print(f"Bronze rows: {bronze_df.count()}")
bronze_df.printSchema()
display(bronze_df.limit(10))

In [0]:
reject_df = (
    bronze_df.filter(~valid_condition)
    .withColumn(
        "reject_reason",
        F.when(~F.col("latitude").between(-90, 90), F.lit("invalid latitude"))
         .when(~F.col("longitude").between(-180, 180), F.lit("invalid longitude"))
         .when(F.col("vessel_id").isNull(), F.lit("missing vessel_id"))
         .when(F.col("event_timestamp").isNull(), F.lit("missing event_timestamp"))
         .otherwise(F.lit("unknown"))
    )
)
print(f"Valid rows: {bronze_df.filter(valid_condition).count()}")

In [0]:
valid_condition = (
    F.col("latitude").between(-90, 90)
    & F.col("longitude").between(-180, 180)
    & F.col("vessel_id").isNotNull()
    & F.col("event_timestamp").isNotNull()
)


In [0]:
silver_df = add_silver_features(
    validate_coordinates(bronze_df.filter(valid_condition))
)


In [0]:
silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_ais_canonical")

reject_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("bronze_ais_rejects")

In [0]:
print(f"Silver rows: {spark.table('silver_ais_events').count()}")
print(f"Reject rows: {spark.table('bronze_ais_rejects').count()}")

spark.table("silver_ais_events").printSchema()
display(spark.table("silver_ais_events").limit(10))
display(spark.table("bronze_ais_rejects").limit(10))